# Agent-B Phase 2.5: Historical FinGPT Sentiment Scoring (17-Year Pipeline)

This notebook implements the exact `ScrapeGraphAI -> FinGPT` pipeline for the full 2007-2026 historical dataset. It uses a lightweight LLM (Qwen) inside ScrapeGraphAI to parse the raw text from FinSen and recent articles, converting them into the mandated JSON dictionary format before passing them to the FinGPT LoRA model for final sentiment scoring.

**Hardware:** Requires T4 GPU
**Est. Runtime:** 4-8 Hours

### Prerequisites
Ensure you have added the following to your Colab **Secrets**:
- `KAGGLE_USERNAME`
- `KAGGLE_KEY`
- `HF_TOKEN`

In [1]:
!pip install -q -U kaggle datasets pandas accelerate bitsandbytes xformers peft transformers huggingface_hub scrapegraphai gnews nest-asyncio python-dotenv langchain-huggingface beautifulsoup4 requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 114.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 109.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!playwright install chromium
!playwright install-deps


175.4 MiB [                    ] 0% 0.0s175.4 MiB [                    ] 0% 129.3s175.4 MiB [                    ] 0% 397.0s175.4 MiB [                    ] 0% 453.7s175.4 MiB [                    ] 0% 365.7s175.4 MiB [                    ] 0% 315.5s175.4 MiB [                    ] 0% 284.6s175.4 MiB [                    ] 0% 263.2s175.4 MiB [                    ] 0% 246.7s175.4 MiB [                    ] 0% 230.9s175.4 MiB [                    ] 0% 216.5s175.4 MiB [                    ] 0% 204.4s175.4 MiB [                    ] 0% 179.7s175.4 MiB [                    ] 0% 161.8s175.4 MiB [                    ] 0% 148.3s175.4 MiB [                    ] 0% 137.3s175.4 MiB [                    ] 0% 131.5s175.4 MiB [                    ] 0% 121.0s175.4 MiB [                    ] 0% 110.2s175.4 MiB [                    ] 0% 100.1s175.4 MiB [                    ] 0% 90.1s175.4 MiB [                    ] 0% 81.6s175.4 MiB [                    ] 0% 75.1s175.4 MiB [                    ] 0% 69.

In [3]:
try:
    # This automatically runs when on Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except ImportError:
    # Fallback just in case you run it locally later
    import os
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing! Please add it to Kaggle Secrets (Add-ons -> Secrets).")

from huggingface_hub import login
login(token=HF_TOKEN)

### 1. Download FinSen Dataset (2007-2023)

In [4]:
import kaggle
import pandas as pd

print("Downloading FinSen Dataset from Kaggle...")
kaggle.api.dataset_download_files('eaglewhl/finsen-financial-sentiment-dataset', path='.', unzip=True)

try:
    df_finsen = pd.read_csv('FinSen_US_Categorized_Timestamp.csv')
except FileNotFoundError:
    df_finsen = pd.read_csv('FinSen_US_Categorized.csv')

print("Columns found:", df_finsen.columns.tolist())

# Lowercase all columns to prevent KeyError on 'Content' vs 'content'
df_finsen.columns = df_finsen.columns.str.lower()

# Identify the date column dynamically
date_col = next((col for col in df_finsen.columns if col in ['date', 'timestamp', 'time', 'datetime', 'published_at', 'publishedat']), None)
if not date_col:
    raise ValueError(f"Could not find a date/timestamp column. Available columns: {df_finsen.columns.tolist()}")

df_finsen['date'] = pd.to_datetime(df_finsen[date_col], errors='coerce', dayfirst=True)
df_finsen = df_finsen.dropna(subset=['date', 'content'])
df_finsen = df_finsen.sort_values('date')
print(f"FinSen Loaded: {len(df_finsen)} rows.")

Dataset URL: https://www.kaggle.com/datasets/eaglewhl/finsen-financial-sentiment-dataset
Columns found: ['Title', 'Tag', 'Time', 'Content']
FinSen Loaded: 15534 rows.


### 2. Scrape Recent URLs via GNews (2023-2026)
We limit fetching to 2 top articles per day to avoid redundancy while retaining the most relevant news.

In [5]:
import json
import os

print("Loading pre-scraped URLs from JSON file...")

# The exact path where Kaggle mounts your dataset
json_path = "/kaggle/input/datasets/rudyxx07/agent-b-training-data/gnews_scraped_urls.json"

with open(json_path, "r") as f:
    saved_data = json.load(f)

urls_to_scrape = saved_data["urls_to_scrape"]
url_to_date = saved_data["url_to_date"]

print(f"Loaded {len(urls_to_scrape)} unique highly-relevant URLs for the recent gap.")

Loading pre-scraped URLs from JSON file...
Loaded 443 unique highly-relevant URLs for the recent gap.


### Step 3: Load Local Models (Qwen & FinGPT)

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import nest_asyncio

nest_asyncio.apply()

# ──────────────────────────────────────────────────────────────
# 3A: Load FinGPT-Llama3 (the sentiment scorer) FIRST
# ──────────────────────────────────────────────────────────────
print("Loading FinGPT-Llama3 into local VRAM...")
base_model = "meta-llama/Meta-Llama-3-8B"
peft_model = "FinGPT/fingpt-mt_llama3-8b_lora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token

quantization_config_fingpt = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization_config_fingpt,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, peft_model)
model.eval()
print("FinGPT Loaded!")

# ──────────────────────────────────────────────────────────────
# 3B: Load Qwen 3.5 0.8B for ScrapeGraphAI (lightweight + fast)
# Only used for URL scraping in Step 4B. 0.8B is sufficient for
# extracting 3 JSON fields from a webpage.
# ──────────────────────────────────────────────────────────────
print("Loading Qwen3.5-0.8B for ScrapeGraphAI...")
qwen_model_id = "Qwen/Qwen3.5-0.8B"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_id)
qwen_quant_config = BitsAndBytesConfig(load_in_4bit=True)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_id,
    quantization_config=qwen_quant_config,
    device_map="auto"
)

qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_new_tokens=256,
    max_length=None,
    temperature=0.01,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=qwen_pipe)
chat_model = ChatHuggingFace(llm=llm)

from scrapegraphai.helpers import models_tokens
models_tokens['huggingface_endpoint'] = {"Qwen/Qwen3.5-0.8B": 8192}
models_tokens['huggingface'] = {"Qwen/Qwen3.5-0.8B": 8192}
models_tokens['Qwen'] = {"Qwen/Qwen3.5-0.8B": 8192}

graph_config = {
    "llm": {
        "model_instance": chat_model,
        "model_tokens": 8192,
    },
    "verbose": False,
}
print("Qwen 0.8B + ScrapeGraphAI Ready!")


Loading FinGPT-Llama3 into local VRAM...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Qwen + ScrapeGraphAI Ready!


### Step 4: Run FinGPT and ScrapeGraphAI on FinSen and URLs

In [7]:
import json
import re
import time
import requests
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime
import pandas as pd

results = []

# ──────────────────────────────────────────────────────────────
# FinGPT Sentiment Scoring Function
# Uses the EXACT prompt template from FinGPT documentation:
#   Instruction: What is the sentiment of this news? ...
#   Input: <text>
#   Answer:
# FinGPT outputs: "positive", "negative", or "neutral"
# We map these to floats: positive=+1.0, neutral=0.0, negative=-1.0
# ──────────────────────────────────────────────────────────────
def run_fingpt(text):
    """Score a piece of text using FinGPT. Returns a float in [-1.0, 1.0]."""
    clean_text = str(text)[:2000].replace("\n", " ").strip()
    if not clean_text:
        return 0.0

    prompt = (
        "Instruction: What is the sentiment of this news? "
        "Please choose an answer from {negative/neutral/positive}\n"
        f"Input: {clean_text}\n"
        "Answer: "
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)

    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip().lower()

    if "positive" in response:
        return 1.0
    elif "negative" in response:
        return -1.0
    else:
        return 0.0

# ──────────────────────────────────────────────────────────────
# Tag-based filtering for FinSen dataset
# ──────────────────────────────────────────────────────────────
print(f"Original FinSen rows: {len(df_finsen)}")

oil_keywords = (
    "(?i)oil|energy|gas|gasoline|commodity|commodities|petroleum|"
    "opec|wti|brent|crude|barrel|refiner|fuel|lng|natural gas|"
    "drilling|pipeline|shale|offshore|upstream|downstream|petrochemical"
)

if 'tag' in df_finsen.columns:
    df_finsen_filtered = df_finsen[df_finsen['tag'].str.contains(oil_keywords, na=False)].copy()
else:
    df_finsen_filtered = df_finsen[df_finsen['content'].str.contains(oil_keywords, na=False)].copy()
print(f"Filtered FinSen rows (Crude Oil related): {len(df_finsen_filtered)}")

# ──────────────────────────────────────────────────────────────
# STEP 4A: Process FinSen — Feed text DIRECTLY to FinGPT
# ──────────────────────────────────────────────────────────────
print("\nStep 4A: Scoring FinSen articles with FinGPT...")
pbar1 = tqdm(df_finsen_filtered.iterrows(), total=len(df_finsen_filtered), desc="FinSen")

for idx, row in pbar1:
    try:
        score = run_fingpt(row['content'])
        date_str = row['date'].strftime('%Y-%m-%d')
        results.append({"date": date_str, "fingpt_sentiment": score})
        pbar1.set_postfix({"Date": date_str, "Score": f"{score:.1f}"})
    except Exception as e:
        pbar1.set_postfix({"Error": str(e)[:40]})
        continue

# ──────────────────────────────────────────────────────────────
# STEP 4B: Process URLs — ScrapeGraphAI with 30s timeout,
# then fallback to requests+BeautifulSoup if it fails.
# ──────────────────────────────────────────────────────────────
from scrapegraphai.graphs import SmartScraperGraph
from pydantic import BaseModel, Field

class CrudeOilSentimentSchema(BaseModel):
    main_announcement: str = Field(description="The main announcement regarding crude oil")
    supporting_metrics: str = Field(description="Any supporting metrics regarding crude oil")
    contradictions_or_risks: str = Field(description="Any contradictions or risks regarding crude oil")

scrape_prompt = """Extract the main announcement, supporting metrics, and any contradictions or risks regarding crude oil from this text.
Output ONLY valid JSON. Do not output any thinking process or explanations."""

def scrape_with_timeout(url, timeout_sec=30):
    """Run ScrapeGraphAI with a hard timeout using ThreadPoolExecutor."""
    def _scrape():
        scraper = SmartScraperGraph(
            prompt=scrape_prompt,
            source=url,
            config=graph_config,
            schema=CrudeOilSentimentSchema
        )
        return scraper.run()

    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_scrape)
        return future.result(timeout=timeout_sec)

def fetch_fallback(url):
    """Plain HTTP fetch as fallback when ScrapeGraphAI fails."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    resp = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(resp.content, "html.parser")
    paragraphs = soup.find_all("p")
    return " ".join([p.text for p in paragraphs if len(p.text) > 30])

print("\nStep 4B: Scraping + scoring recent URLs...")
pbar2 = tqdm(urls_to_scrape, desc="Recent News")

for i, url in enumerate(pbar2):
    summary_text = ""

    # Attempt 1: ScrapeGraphAI with Qwen (30s timeout)
    try:
        extracted = scrape_with_timeout(url, timeout_sec=30)
        if isinstance(extracted, dict):
            parts = [str(v) for v in extracted.values() if v and v != "NA"]
            summary_text = " ".join(parts) if parts else ""
        elif isinstance(extracted, str):
            summary_text = extracted
    except Exception:
        pass  # Fall through to fallback

    # Attempt 2: Plain HTTP fetch
    if not summary_text:
        try:
            summary_text = fetch_fallback(url)
        except Exception:
            summary_text = ""

    # Skip if we got nothing at all
    if not summary_text:
        pbar2.set_postfix({"Status": "Skip (no content)"})
        continue

    # Score with FinGPT
    score = run_fingpt(summary_text)

    article_date = url_to_date.get(url, "")
    try:
        parsed_date = pd.to_datetime(article_date).strftime('%Y-%m-%d')
    except:
        parsed_date = datetime.now().strftime('%Y-%m-%d')

    results.append({"date": parsed_date, "fingpt_sentiment": score})
    pbar2.set_postfix({"Date": parsed_date, "Score": f"{score:.1f}"})

# ──────────────────────────────────────────────────────────────
# STEP 5: Aggregate and Save
# ──────────────────────────────────────────────────────────────
if len(results) == 0:
    print("\nERROR: No results were produced! Check logs above.")
else:
    df_final = pd.DataFrame(results)
    df_final = df_final.groupby('date')['fingpt_sentiment'].mean().reset_index()
    df_final = df_final.sort_values('date')
    df_final.to_csv('fingpt_historical_scores.csv', index=False)
    print(f"\nFINISHED! Saved {len(df_final)} daily scores to fingpt_historical_scores.csv")
    print(df_final.head(10))


Original FinSen rows: 15534
Filtered FinSen rows (Crude Oil related): 646

Step 4A: Scoring FinSen articles with FinGPT...


FinSen:   0%|          | 0/646 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
FinSen: 100%|██████████| 646/646 [13:16<00:00,  1.23s/it, Date=2023-07-13, Score=-1.0]



Step 4B: Scraping + scoring recent URLs...


Recent News:   0%|          | 0/443 [00:00<?, ?it/s]--- Executing Fetch Node ---
--- (Fetching HTML from: https://news.google.com/rss/articles/CBMiiwFBVV95cUxQNmc0YnRWZXpQajZ2eW1ZdWNrUVhDaVRQck9oNm54TjhBOVBQR3JKV0VyZUxZTnBuWnFfNVh2T3I4amVBUlQzeEJBZ0tXUVdlWEhwV285dHAtcVVXVWlGTVF4Z1JQQnduWkkwVERWb3lfNDNjM1VZdUw5MEZWUk1Rb19zR0pXUnVhVTB3?oc=5&hl=en-SG&gl=SG&ceid=SG:en) ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Error during chain execution: Invalid json output: Thinking Process:

1.  **Analyze the Request:**
    *   Role: Website scraper.
    *   Inp

KeyboardInterrupt: 